In [6]:
import sys
sys.path.append('/home/jovyan/BenchmarkingML4KGE_extraction')
import os
from pathlib import Path

from utils.XMLParser import XMLParser
from utils import experimentation_utils
from utils.experimentation_utils import get_process_info, get_gpu_usage, calcular_bertscore_listas
import json
import requests
import torch
from tqdm.auto import tqdm

import time
from bert_score import score
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()


from gliner import GLiNER
from utils.grobid_service import GrobidService

import ollama
from ollama import Client, ResponseError
from utils import llm_preprocessing
from transformers import AutoModelForCausalLM, AutoTokenizer
import accelerate

import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk import sent_tokenize
nltk.download('punkt_tab')
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter

with open('../data/dataset_con_rutas_xml.json', 'r', encoding='utf-8') as f:
    kge_dataset=json.load(f)

service=GrobidService()

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
device = "cuda" if torch.cuda.is_available() else "cpu"

def extract_with_gliner(text: str, labels: list):
    entities=gliner_model.predict_entities(text,labels,threshold=0.4)
    predictions=list(set(ent['text'] for ent in entities))
    return predictions

/home/jovyan/.local/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [8]:
model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
max_context_tokens = 32768 - 2048


def extract_model_qwen(text):
    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]
    question="What is the name of the model presented in this paper?"
    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no model presented")
    chat.append({"role":"user","content":prompt})
    predictions=llm_preprocessing.query_model_return_list(model,chat,tokenizer,local=True)

    return predictions


/home/jovyan/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [9]:
model_name="llama3"

def extract_implementation_llama(text):
    client=Client(timeout=600.0)
    question="Is there a URL in the paper providing the implementation of the model?"
    chat = [
        {"role": "system", "content":
            "You are an assistant for QA tasks. Use only provided context."
        },
        {"role": "user", "content": f"Context chunk: {text}"}
    ]

    prompt=(f"Given the following question: {question}"+"Return the answer only in a Python list format, i.e. ['A','B']. You must return an empty list if there is no implementation")
    chat.append({"role":"user","content":prompt})
    try:
        response=client.chat(model=model_name, messages=chat)
        predictions=response['message']['content']
    except Exception as e:
        print(f"Timeout error")
        

    return predictions
    

In [10]:
import joblib

def create_chunks(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def train_and_save_classifier():
    resultados = []
    tiempos = []
    scores_f1 = []
    
    base_path = Path(os.getcwd()).parent
    
    X_chunks=[]
    Y_chunks=[]
    paper_ids=[]
    
    for i,paper in enumerate(kge_dataset):
        category=paper.get('category',[])
        xml_path=paper.get('xml_file')
        if not xml_path:
            continue
        archive=Path(xml_path)
        filename=archive.name
        filename=filename.replace("\\","/")
    
        base_path = Path(os.getcwd()).parent
        xml_path = base_path / "data" / filename
    
        parser=XMLParser(xml_path)
        text=parser.get_full_text()
    
        if text and category:
            fragments=create_chunks(text)
            for frag in fragments:
                X_chunks.append(frag)
                Y_chunks.append(category)
                paper_ids.append(i)
                
    print(f"Generated a total of {len(X_chunks)}")
    
    
    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X_chunks, Y_chunks, paper_ids, test_size=0.2, random_state=42, stratify=Y_chunks
    )
    # 4. Definir el Pipeline
    # Usamos stop_words para limpiar ruido y subimos ngram_range para capturar conceptos compuestos (ej: "machine learning")
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words='english', 
            max_features=10000, 
            ngram_range=(1, 2)
        )),
        ('clf', LinearSVC(C=1.0, class_weight='balanced', random_state=42))
    ])
    
    
    # 5. Entrenar
    pipeline.fit(X_train, y_train)
    
    y_pred_chunks = pipeline.predict(X_test)
    
    # 2. Agrupamos predicciones por Paper ID
    paper_votes = {}
    paper_ground_truth = {}
    
    for i in range(len(ids_test)):
        p_id = ids_test[i]
        pred = y_pred_chunks[i]
        real = y_test[i]
        
        if p_id not in paper_votes:
            paper_votes[p_id] = []
            paper_ground_truth[p_id] = real
        
        paper_votes[p_id].append(pred)
    
    # 3. Calculamos el ganador (Voto Mayoritario) por paper
    final_paper_preds = []
    final_paper_real = []
    
    for p_id, votes in paper_votes.items():
        voto_ganador = Counter(votes).most_common(1)[0][0]
        final_paper_preds.append(voto_ganador)
        final_paper_real.append(paper_ground_truth[p_id])
    
    path = base_path / "pipeline_clasificador_kge.joblib"
    joblib.dump(pipeline, path)
    return path
    
pipeline_path = train_and_save_classifier()

def extract_classification(text):
    classification_pipeline = joblib.load(pipeline_path)
    predicted_class = classification_pipeline.predict([text])
    return predicted_class

Generated a total of 1408


In [11]:
dataset_reducido = kge_dataset[:5]

In [21]:
paper_evaluation_data = []
tiempos = []

dataset_ext_scores = []
task_ext_scores = []
model_ext_scores = []
implementation_ext_scores = []
taxonomy_ext_scores = []

useful_f1_scores = []

base_path = Path(os.getcwd()).parent

for paper in tqdm(dataset_reducido, desc="Processing"):
    xml_path = paper.get('xml_file')

    # Extraer el groundtruth de cada campo para este paper
    task_gt = paper.get('tasks',[])
    dataset_gt = paper.get('Datasets',[])
    model_gt = []
    method_list = paper["methods"]
    for method in method_list:
        method_name = method["name"]
        model_gt.append(method_name)
    implementation_gt = paper.get('repo_url','')
    category_gt=paper.get('category',[])
    
    if not xml_path:
        continue
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")
    base_path = Path(os.getcwd()).parent
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    abstract=parser.get_abstract()
    full_text=parser.get_full_text()

    if not abstract or not full_text:
        continue
        print('An error occurred while processing')

    init_time=time.time()

    extracted_datasets = extract_with_gliner(abstract, ['dataset'])
    extracted_tasks = extract_with_gliner(abstract, ['task'])
    extracted_models = extract_model_qwen(abstract)
    extracted_implementations = extract_implementation_llama(abstract)
    extracted_category = extract_classification(full_text)
    print(extracted_category)

    mem_usage, cpu_usage = get_process_info()
    vram_usage = get_gpu_usage()
    end_time = time.time()
    
    total_time = end_time - init_time
    tiempos.append(total_time)

    dataset_f1 = experimentation_utils.calcular_bertscore_listas(extracted_datasets,dataset_gt)
    dataset_ext_scores.append(dataset_f1)
    
        
    task_f1 = experimentation_utils.calcular_bertscore_listas(extracted_tasks,task_gt)
    task_ext_scores.append(task_f1)
    model_f1 = experimentation_utils.calcular_bertscore_listas(extracted_models,model_gt)
    model_ext_scores.append(model_f1)
    implementation_f1 = experimentation_utils.calcular_bertscore_listas(extracted_implementations,implementation_gt)
    implementation_ext_scores.append(implementation_f1)
    taxonomy_f1 = experimentation_utils.calcular_bertscore_listas(extracted_category,category_gt)
    taxonomy_ext_scores.append(taxonomy_f1)

    average_f1 = (dataset_f1 + task_f1 + model_f1 + implementation_f1 + taxonomy_f1)/5

    paper_evaluation_data.append({
            "title": paper.get('title'),
            "time": total_time,
            "vram_usage": vram_usage,
            "ram_mem_usage": mem_usage,
            "cpu_usage": cpu_usage,
            "extracted_datasets": extracted_datasets,
            "datasets_gt": dataset_gt,
            "dataset_f1": dataset_f1,
            "extracted_tasks": extracted_tasks,
            "tasks_gt": task_gt,
            "task_f1": task_f1,
            "extracted_models": extracted_models,
            "model_gt": model_gt,
            "model_f1": model_f1,
            "extracted_implementations": extracted_implementations,
            "implementation_gt": implementation_gt,
            "implementation_f1": implementation_f1,
            "extracted_category": extracted_category,
            "category_gt": category_gt,
            "taxonomy_f1": taxonomy_f1,
            "average_f1": average_f1,
    })
    #print((sum(dataset_ext_scores.tolist()) / len(dataset_ext_scores.tolist())))
    #print(sum(task_ext_scores.tolist()) / len(task_ext_scores.tolist()))
    #print(sum(model_ext_scores.tolist()) / len(model_ext_scores.tolist()))
    #print(sum(implementation_ext_scores.tolist()) / len(implementation_ext_scores.tolist()))
    #print(sum(taxonomy_ext_scores.tolist()) / len(taxonomy_ext_scores.tolist()))

average_time = sum(tiempos) / len(tiempos) if tiempos else 0

evaluation_data = {
    "total_time": sum(tiempos),
    "average_time": average_time,

    "paper_evaluation_data": paper_evaluation_data
}

with open("efficient_extraction_results.json", "w", encoding="utf-8") as f:
		json.dump(evaluation_data, f, indent=4)

"""
    "average_dataset_f1" : (sum(dataset_ext_scores) / len(dataset_ext_scores)),
    "average_task_f1" : (sum(task_ext_scores) / len(task_ext_scores)),
    "average_model_f1" : (sum(model_ext_scores) / len(model_ext_scores)),
    "average_implementation_f1" : (sum(implementation_ext_scores) / len(implementation_ext_scores)),
    "average_taxonomy_f1" : (sum(taxonomy_ext_scores) / len(taxonomy_ext_scores)),
"""

Processing:   0%|          | 0/5 [00:00<?, ?it/s]

['Semantic matching models']
Error leyendo GPU: Unknown Error


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

['Semantic matching models']
Error leyendo GPU: Unknown Error


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

TypeError: Object of type ndarray is not JSON serializable